In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv("../data/de_day_ahead_prices_2022_2024.csv", index_col="datetime_utc", parse_dates=True)
df.head()

In [ ]:
df.describe()

In [ ]:
df['price_eur_mwh'].plot(figsize=(12, 6), title="DE Day-Ahead Prices (2022)", ylabel="Price (EUR/MWh)") 

In [ ]:
#check timestamp continuity
df.index.is_monotonic_increasing

In [ ]:
df.index.to_series().diff().value_counts()

the three days with 2 hours difference is due to the DST changes, hence the time contintuity is maintainted.

In [ ]:
df[df['price_eur_mwh'] < 0].describe()

### Negative Prices Mean Reversion
How fast do the negative prices revert back?

In [ ]:
# Check mean reversion for all prices
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(df["price_eur_mwh"], lags=48)

Event Study

What happens after a negative price?

In [ ]:
horizons = [1, 3, 6, 12, 24]
neg_times = df[df["price_eur_mwh"] < 0].index

results = {h: [] for h in horizons}

for t in neg_times:
    for h in horizons:
        future_t = t + pd.Timedelta(hours=h)
        if future_t in df.index:
            results[h].append(df.loc[future_t, "price_eur_mwh"])

summary = pd.DataFrame({
    "horizon_hours": horizons,
    "mean_price": [pd.Series(results[h]).mean() for h in horizons],
    "median_price": [pd.Series(results[h]).median() for h in horizons],
    "n_events": [len(results[h]) for h in horizons],
})
summary


In [ ]:
# Control for the hour of the day
hourly_avg = df.groupby(df.index.hour)["price_eur_mwh"].mean()

results_relative = {h: [] for h in horizons}
for t in neg_times:
    for h in horizons:
        future_t = t + pd.Timedelta(hours=h)
        if future_t in df.index:
            actual = df.loc[future_t, "price_eur_mwh"]
            typical_for_that_hour = hourly_avg[future_t.hour]
            results_relative[h].append(actual - typical_for_that_hour)

summary_relative = pd.DataFrame({
    "horizon_hours": horizons,
    "mean_deviation": [pd.Series(results_relative[h]).mean() for h in horizons],
    "median_deviation": [pd.Series(results_relative[h]).median() for h in horizons],
})
summary_relative


### Seasonality Analysis

In [ ]:
# Patterns
hourly_pattern = df.groupby(df.index.hour)["price_eur_mwh"].agg(["mean", "median", "std"])
hourly_pattern.plot(y="mean", marker="o", figsize=(10,4), title="Average price by hour of day")

monthly_pattern = df.groupby(df.index.month)["price_eur_mwh"].agg(["mean", "median", "std"])
monthly_pattern.plot(y="mean", marker="o", figsize=(10,4), title="Average price by month")


#### Seasonal Trend decomposition using LOESS

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(df["price_eur_mwh"], period=24, robust=True)
result = stl.fit()

fig = result.plot()
fig.set_size_inches(12, 8)


### Crisis Era vs Normal Era

In [ ]:
crisis = df["2022-01-01":"2023-02-28"]
normal = df["2023-03-01":"2024-12-31"]

for name, subset in [("crisis", crisis), ("normal", normal)]:
    print(name)
    print(subset["price_eur_mwh"].describe())
    print("negative hours:", (subset["price_eur_mwh"] < 0).sum(), "/", len(subset))
